# Stage 1 — Binary Classifier (Defect vs Normal) — Pipeline Walkthrough

Notebook ini **bukan cara utama buat training** — itu tetap lewat CLI:

```bash
python -m src.train_classification --config configs/classification/cls_stage1_binary.yaml
```

Notebook ini jalanin ulang persis logic yang sama, langkah demi langkah,
pakai fungsi ASLI dari `src/` (bukan reimplementasi) — tujuannya biar kamu
bisa baca & paham apa yang kejadian di tiap tahap, sambil liat output/
visualisasi antara di tiap langkah.

**Prasyarat sebelum cell di bawah bisa jalan beneran:**
1. Dataset stage 1 udah di-build: `python scripts/build_stage1_binary.py ...`
2. `pip install -r requirements.txt`

Kalau prasyaratnya belum ada, cell dari bagian "3. Dataset & DataLoader"
ke bawah bakal error pas load data — itu wajar, notebook ini emang ditulis
duluan sebagai referensi baca, sebelum datanya jadi.

Ada flag `RUN_TRAINING_LOOP` di bagian 8 — default `False` biar buka
notebook ini nggak sengaja mancing training beneran (bisa lama). Ubah ke
`True` kalau emang mau coba jalanin demo training beberapa epoch.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]  # notebooks/classification/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import torch
import matplotlib.pyplot as plt

from src.data_loader import get_classification_dataloaders, IMAGENET_MEAN, IMAGENET_STD
from src.models.classification import build_model
from src.utils.seed import set_seed
from src.utils.class_weights import compute_class_weights
from src.utils.quantization import is_qat_supported, QAT_SUPPORTED_ARCHITECTURES
from src.train_classification import build_optimizer, train_one_epoch, evaluate

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 1. Load config

Semua hyperparameter, path dataset, dan pilihan arsitektur ada di 1 file
yaml — bukan hardcoded di kode. Ganti config = ganti eksperimen tanpa
nyentuh `src/`.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "classification" / "cls_stage1_binary.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, sort_keys=False))


## 2. Reproducibility

`set_seed` ngunci random state python/numpy/torch (CPU & GPU) + matiin
non-determinism cuDNN, biar hasil training bisa direproduksi (bukan buat
naikin performa).

In [ ]:
set_seed(config.get("seed", 42))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 3. Dataset & DataLoader

`FolderClassificationDataset` (custom, BUKAN `torchvision.datasets.ImageFolder`)
baca gambar dari `root/{train,val,test}/<class_name>/*.jpg`. Bedanya sama
`ImageFolder` bawaan: urutan label index-nya DIPAKSA ngikutin urutan
`class_names` di config, bukan alfabetis — penting biar konsisten sama
`class_mapping.py` dan class-weight computation.

Augmentasi (`build_classification_transforms`) beda antara train (random
crop/flip/rotate/color-jitter) vs val/test (cuma resize+center-crop, tanpa
randomness) — standar praktik biar evaluasi konsisten antar run.

In [ ]:
data = get_classification_dataloaders(config)
train_loader, val_loader, test_loader = data["train_loader"], data["val_loader"], data["test_loader"]

print(f"Train: {len(data['train_dataset'])} gambar")
print(f"Val:   {len(data['val_dataset'])} gambar")
print(f"Test:  {len(data['test_dataset'])} gambar")


In [ ]:
def denormalize(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)


images, labels = next(iter(train_loader))
class_names = config["dataset"]["class_names"]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(class_names[label.item()])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Model — backbone + head

`build_model(arch_name, num_classes, head_cfg=...)` pasang backbone
pretrained ImageNet (dari `timm`) + head classifier di atasnya.

Config ini pakai `head.type: linear` (1 Linear layer standar) — karena
dataset stage 1 udah besar & balanced by design (~10.9k gambar, hasil
`build_stage1_binary.py`), beda dari skenario NEU-CLS/X-SDD yang kecil dan
butuh head MLP+dropout sebagai regularizer (bandingkan sama
`configs/classification/cls_neu.yaml`).

In [ ]:
arch_name = config["architectures"][0]  # contoh: resnet18
model = build_model(
    arch_name,
    config["dataset"]["num_classes"],
    pretrained=True,
    head_cfg=config.get("head"),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Arsitektur: {arch_name}")
print(f"Total parameter: {n_params:,}")


## 5. Loss function

`loss.weighted: false` di config ini — karena dataset udah di-stratified
balance ~1:1 (Defect:Normal) dari sononya lewat `build_stage1_binary.py`,
beda dari skenario combined 20-kelas yang WAJIB `weighted: true` (kelas
hasil merge kayak Inclusion/Scratches punya sampel jauh lebih banyak).
`label_smoothing` tetap dipakai sebagai regularizer ringan.

In [ ]:
loss_cfg = config.get("loss", {})
weight = None
if loss_cfg.get("weighted", False):
    weight = compute_class_weights(data["train_dataset"].labels, config["dataset"]["num_classes"]).to(device)
    print(f"Class weights: {weight}")
else:
    print("loss.weighted=false -> CrossEntropyLoss tanpa weight (dataset udah balanced)")

criterion = torch.nn.CrossEntropyLoss(weight=weight, label_smoothing=loss_cfg.get("label_smoothing", 0.0))


## 6. Optimizer & scheduler

In [ ]:
optimizer = build_optimizer(model, config.get("optimizer", {}))
print(optimizer)


## 7. Quantization-aware training (QAT) — opsional

Kalau `quantization.enabled: true` DAN arsitekturnya didukung
(`QAT_SUPPORTED_ARCHITECTURES` — CNN yang FX-traceable), training jalan
fp32 normal dulu sampai epoch `qat_start_epoch`, BARU fake-quant
diaktifkan buat sisa epoch (simulasiin noise int8 SELAMA training, bukan
cuma di akhir). Ini paling relevan buat stage 1 — dia yang paling butuh
ringan (gate classifier, jalan di SETIAP gambar sebelum keputusan lanjut
ke detector atau nggak).

Detail lengkap mekanismenya (freeze observer, freeze BN stats, convert ke
int8 asli di akhir) ada di `src/utils/quantization.py`, di-orchestrate
`src/train_classification.py::train_one_architecture` — notebook ini cuma
nunjukin config-nya, eksekusi penuhnya cek lewat CLI.

In [ ]:
quant_cfg = config.get("quantization", {})
print(f"QAT enabled di config: {quant_cfg.get('enabled', False)}")
print(f"Arsitektur '{arch_name}' didukung QAT: {is_qat_supported(arch_name)}")
print(f"Arsitektur yang didukung QAT: {QAT_SUPPORTED_ARCHITECTURES}")
print(f"Backend: {quant_cfg.get('backend')}, mulai aktif epoch: {quant_cfg.get('qat_start_epoch')}")


## 8. Training loop

Loop asli (`train_one_architecture` di `src/train_classification.py`)
tiap epoch: `train_one_epoch` (forward+backward+step) → `evaluate` (hitung
metric di val set) → simpan checkpoint kalau metric monitor
(`early_stopping.metric`, di sini `recall_per_class.Defect`) membaik →
cek early stopping.

Sel di bawah jalanin versi ringkas-nya (beberapa epoch aja, buat demo
baca) pakai fungsi ASLI yang sama (`train_one_epoch`/`evaluate`), bukan
reimplementasi. Buat training penuh (semua arsitektur, semua epoch, +QAT,
+logging MLflow lengkap), pakai CLI:

```bash
python -m src.train_classification --config configs/classification/cls_stage1_binary.yaml
```

In [ ]:
RUN_TRAINING_LOOP = False  # ubah ke True kalau mau coba jalanin demo training beneran
DEMO_EPOCHS = 2

if RUN_TRAINING_LOOP:
    for epoch in range(1, DEMO_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_result = evaluate(model, val_loader, device, class_names)
        recall_defect = val_result["scalars"]["recall_per_class.Defect"]
        print(f"epoch {epoch}: train_loss={train_loss:.4f} val_recall_Defect={recall_defect:.4f}")
else:
    print("RUN_TRAINING_LOOP=False, skip eksekusi. Set True buat coba training beneran (butuh dataset + waktu).")


## 9. Evaluasi & confusion matrix

`evaluate()` balikin dict berisi semua metric (accuracy, precision/recall/f1
macro & weighted + per-class, AUC) + confusion matrix + classification
report text — persis yang di-log ke MLflow di training asli.

In [ ]:
if RUN_TRAINING_LOOP:
    test_result = evaluate(model, test_loader, device, class_names)
    print(test_result["classification_report"])

    cm = test_result["confusion_matrix"]
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names)
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
    plt.show()
else:
    print("Skip - RUN_TRAINING_LOOP masih False, belum ada model yang beneran ke-training.")


## 10. Yang di-log ke MLflow (di run CLI asli)

Training run yang sebenarnya (lewat CLI) log semua ini per arsitektur
(1 MLflow run = 1 arsitektur):

- **Semua hyperparameter** dari config (flattened)
- **Metric per-epoch**: train_loss, lr, val accuracy/precision/recall/f1
  (macro + weighted + per-class), val AUC, status QAT aktif/belum
- **Metric final test set** + confusion matrix (image artifact) +
  classification report (text artifact)
- **Best checkpoint** (`best.pt`) berdasarkan `recall_per_class.Defect`
- Kalau QAT aktif: metric model quantized (`test_quantized_*`),
  perbandingan ukuran (`fp32_model_size_mb` vs `quantized_model_size_mb`,
  `model_size_reduction_pct`), checkpoint int8 terpisah (`best_quantized.pt`)

Detail lengkapnya ada di `src/utils/mlflow_utils.py` dan
`src/train_classification.py::train_one_architecture`.